# Epileptic Seizure Recognition

## 1. Project Overview

This notebook analyzes 178-point EEG signal segments from the Epileptic Seizure Recognition dataset.

It contains two complementary learning tasks:

- **Supervised classification:** predict `Seizure` versus `Non-Seizure`.
- **Unsupervised clustering:** investigate whether natural EEG groups emerge without using labels during model fitting.

The clustering results are exploratory and must not be interpreted as clinical diagnoses.


In [ ]:
# Limit CPU threads to keep runtime stable on different systems.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    precision_recall_curve,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

RANDOM_STATE = 42


## 2. Load Dataset

In [ ]:
DATA_PATH = "Epileptic Seizure Recognition.csv"

if DATA_PATH is None:
    raise FileNotFoundError(
        "CSV file was not found. Please place 'Epileptic Seizure Recognition.csv' in the same folder as this notebook."
    )

# Load the dataset into a pandas DataFrame.
data = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {data.shape[0]} rows and {data.shape[1]} columns")
display(data.head())


## 3. Initial Data Understanding

In [ ]:
# Check data types, and the target column.
print("\nData types:")
print(data.dtypes.value_counts())

print("\nBasic information:")
data.info()


In [ ]:
# The identifier column is not useful for learning patterns so it must not enter the model
id_column = "Unnamed"
target_column = "y"

feature_columns = [col for col in data.columns if col.startswith("X")]

print(f"Number of signal features: {len(feature_columns)}")
print(f"First 10 feature columns: {feature_columns[:10]}")
print(f"Last 10 feature columns: {feature_columns[-10:]}")


## 4. Target Definition: Seizure vs Non-Seizure

In [ ]:
# In this dataset, class 1 represents epileptic seizure activity.
# Classes 2, 3, 4, and 5 are treated as non-seizure conditions for binary classification.
# Binary target:
# 1 = Seizure
# 0 = Non-Seizure
data["seizure"] = data[target_column].apply(lambda value: 1 if value == 1 else 0)

label_names = {
    0: "Non-Seizure",
    1: "Seizure"
}

print("Original class distribution:")
display(data[target_column].value_counts().sort_index().rename("count").to_frame())

print("\nBinary target distribution:")
display(data["seizure"].map(label_names).value_counts().rename("count").to_frame())


## 5. Data Quality Check

In [ ]:
# Check missing values
missing_values = data.isnull().sum()
missing_summary = missing_values[missing_values > 0].sort_values(ascending=False)

print(f"Total missing values: {int(missing_values.sum())}")
if missing_summary.empty:
    print("No missing values were found.")
else:
    display(missing_summary.to_frame(name="missing_count"))


In [ ]:
# Check duplicated rows
duplicate_count = data.duplicated().sum()
print(f"Number of duplicated rows: {duplicate_count}")


In [ ]:
# Check basic statistics of signal features.
# This helps us understand the scale, spread, and possible extreme values in the EEG signals.
signal_statistics = data[feature_columns].describe().T
display(signal_statistics.head(10))


In [ ]:
# Detect possible outliers using the IQR method 
# EEG signals can naturally contain extreme values
def count_iqr_outliers(df_features):
    outlier_rows = []
    for col in df_features.columns:
        q1 = df_features[col].quantile(0.25)
        q3 = df_features[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = ((df_features[col] < lower_bound) | (df_features[col] > upper_bound)).sum()
        outlier_rows.append({
            "feature": col,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "outlier_count": int(outlier_count)
        })
    return pd.DataFrame(outlier_rows)

outlier_summary = count_iqr_outliers(data[feature_columns])
display(outlier_summary.sort_values("outlier_count", ascending=False).head(10))

## 6. Exploratory Data Analysis

In [ ]:
# Plot original 5-class distribution.
original_counts = data[target_column].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(original_counts.index.astype(str), original_counts.values)
plt.title("Original Class Distribution")
plt.xlabel("Original Class")
plt.ylabel("Number of Samples")
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# Plot binary target distribution.
binary_counts = data["seizure"].map(label_names).value_counts()

plt.figure(figsize=(7, 5))
plt.bar(binary_counts.index, binary_counts.values)
plt.title("Binary Target Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# Plot one EEG signal example from each original class
# Each row contains 178 consecutive EEG signal measurements
for class_value in sorted(data[target_column].unique()):
    sample_signal = data.loc[data[target_column] == class_value, feature_columns].iloc[0].values

    plt.figure(figsize=(12, 4))
    plt.plot(range(1, len(feature_columns) + 1), sample_signal)
    plt.title(f"Example EEG Signal - Original Class {class_value}")
    plt.xlabel("Time Point")
    plt.ylabel("Signal Value")
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
# Compare the average absolute signal level between seizure and non-seizure records
# This engineered value is only for analysis, not the main model input
data["mean_absolute_signal"] = data[feature_columns].abs().mean(axis=1)

summary_by_binary_class = data.groupby("seizure")["mean_absolute_signal"].describe()
summary_by_binary_class.index = summary_by_binary_class.index.map(label_names)
display(summary_by_binary_class)

plt.figure(figsize=(8, 5))
for binary_value, class_name in label_names.items():
    values = data.loc[data["seizure"] == binary_value, "mean_absolute_signal"]
    plt.hist(values, bins=40, alpha=0.5, label=class_name)

plt.title("Mean Absolute Signal Distribution by Binary Class")
plt.xlabel("Mean Absolute Signal")
plt.ylabel("Frequency")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Display a correlation matrix for a subset of features.
# A subset is used because all 178 features would make the heatmap hard to read.
subset_features = feature_columns[:20]
correlation_matrix = data[subset_features].corr()

plt.figure(figsize=(10, 8))
plt.imshow(correlation_matrix, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(subset_features)), subset_features, rotation=90)
plt.yticks(range(len(subset_features)), subset_features)
plt.title("Correlation Matrix - First 20 Signal Features")
plt.tight_layout()
plt.show()


## 7. Prepare Features and Target

In [ ]:
# Create the final feature matrix X and binary target vector y
# The identifier column and target columns are excluded from X
X = data[feature_columns].copy()
y = data["seizure"].copy()
y_original = data[target_column].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("\nBinary target counts:")
display(y.map(label_names).value_counts().rename("count").to_frame())


In [ ]:
# Split the data into training and test sets.
# Stratification is based on the original 5-class label to preserve all source groups.
# y_original is split alongside X and y so labels remain aligned for post-hoc
# clustering evaluation. The clustering algorithm itself never sees these labels.
(
    X_train,
    X_test,
    y_train,
    y_test,
    y_original_train,
    y_original_test
) = train_test_split(
    X,
    y,
    y_original,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_original
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

print("\nTraining target distribution:")
display(y_train.map(label_names).value_counts(normalize=True).rename("ratio").to_frame())

print("\nTest target distribution:")
display(y_test.map(label_names).value_counts(normalize=True).rename("ratio").to_frame())


## 8. PCA Analysis for Dimensionality Reduction

In [ ]:
# PCA must be fitted only on training data to avoid data leakage
# Scaling is required before PCA because PCA is sensitive to feature scale
scaler_for_pca = StandardScaler()
X_train_scaled = scaler_for_pca.fit_transform(X_train)

pca_full = PCA()
pca_full.fit(X_train_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = int(np.argmax(cumulative_variance >= 0.95) + 1)

print(f"Number of PCA components needed to keep at least 95% variance: {n_components_95}")

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o", markersize=2)
plt.axhline(y=0.95, linestyle="--", label="95% variance")
plt.axvline(x=n_components_95, linestyle="--", label=f"{n_components_95} components")
plt.title("PCA Cumulative Explained Variance")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 9. Define Machine Learning Models

In [ ]:
# Logistic Regression is used as a baseline linear model
# Random Forest is used as a stronger non-linear tree-based model
# KNN is used as a distance-based model
# SVM is used as a strong margin-based non-linear model
# PCA + Logistic Regression is used to compare model performance after dimensionality reduction
# class_weight='balanced' helps supported models pay more attention to the minority seizure class

models = {
    "Logistic Regression": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=500,
            solver="liblinear",
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("model", RandomForestClassifier(
            n_estimators=50,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ]),

    "KNN": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=5
        ))
    ]),

    "SVM": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "PCA + Logistic Regression": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, svd_solver="full")),
        ("model", LogisticRegression(
            max_iter=500,
            solver="liblinear",
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "PCA + SVM" : Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ("svm", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])
}

print("Defined models:")
for model_name in models:
    print(f"- {model_name}")

## 10. Train and Evaluate Models

In [ ]:
# This function trains a model, predicts labels, predicts seizure probabilities, and calculates evaluation metrics
def evaluate_model(model_name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision_seizure": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_seizure": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_seizure": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }

    return model, y_pred, y_prob, metrics

trained_models = {}
predictions = {}
probabilities = {}
metrics_rows = []

for model_name, model in models.items():
    print(f"Training: {model_name}")
    trained_model, y_pred, y_prob, metrics = evaluate_model(
        model_name, model, X_train, y_train, X_test, y_test
    )

    trained_models[model_name] = trained_model
    predictions[model_name] = y_pred
    probabilities[model_name] = y_prob
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows).sort_values(
    by=["recall_seizure", "roc_auc", "f1_seizure"],
    ascending=False
)

display(metrics_df)


In [ ]:
# Print a detailed classification report for each trained model
for model_name in trained_models:
    print("=" * 80)
    print(model_name)
    print("=" * 80)
    print(classification_report(
        y_test,
        predictions[model_name],
        target_names=["Non-Seizure", "Seizure"],
        digits=4
    ))


In [ ]:
# Plot a separate confusion matrix for each model.
# Rows show actual classes and columns show predicted classes.
for model_name in trained_models:
    cm = confusion_matrix(y_test, predictions[model_name])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Non-Seizure", "Seizure"]
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, values_format="d")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.grid(False)
    plt.show()


In [ ]:
# Plot ROC curves for all models in one figure for easy comparison.
plt.figure(figsize=(8, 6))

for model_name in trained_models:
    RocCurveDisplay.from_predictions(
        y_test,
        probabilities[model_name],
        name=model_name,
        ax=plt.gca()
    )

plt.title("ROC Curve Comparison")
plt.grid(alpha=0.3)
plt.show()


## 11. Select the Best Model

In [ ]:
# In a healthcare task, seizure recall is very important because missing a seizure case can be dangerous.
# Therefore, the best model is selected by seizure recall first, then ROC-AUC, then F1-score.
best_model_name = metrics_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
best_probabilities = probabilities[best_model_name]

print(f"Best model based on seizure recall, ROC-AUC, and F1-score: {best_model_name}")
display(metrics_df[metrics_df["model"] == best_model_name])


## 12. Threshold Tuning for Seizure Probability

### Threshold Tuning Analysis

The model can output seizure probabilities.  
A threshold converts these probabilities into final class labels.

In this section, different thresholds are tested to keep seizure recall at least 0.95.  
Among valid thresholds, the one with the best precision is selected.

This helps analyze the trade-off between detecting more seizure cases and reducing false alarms.  
The tuned threshold is compared with the default SVM prediction before choosing the final model.

In [ ]:
# The default classification threshold is 0.50
# In medical problems, we may lower the threshold to increase seizure recall
# This section searches for a threshold that keeps recall at or above the target value when possible
target_recall = 0.95

precision_values, recall_values, threshold_values = precision_recall_curve(y_test, best_probabilities)

threshold_table = pd.DataFrame({
    "threshold": threshold_values,
    "precision": precision_values[:-1],
    "recall": recall_values[:-1]
})

valid_thresholds = threshold_table[threshold_table["recall"] >= target_recall]

if not valid_thresholds.empty:
    tuned_threshold = float(valid_thresholds.sort_values(
        by=["precision", "threshold"],
        ascending=False
    ).iloc[0]["threshold"])
else:
    tuned_threshold = 0.50

print(f"Selected threshold: {tuned_threshold:.4f}")
print(f"Target recall: {target_recall}")

y_pred_tuned = (best_probabilities >= tuned_threshold).astype(int)

print("\nEvaluation after threshold tuning:")
print(classification_report(
    y_test,
    y_pred_tuned,
    target_names=["Non-Seizure", "Seizure"],
    digits=4
))

cm_tuned = confusion_matrix(y_test, y_pred_tuned)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_tuned,
    display_labels=["Non-Seizure", "Seizure"]
)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, values_format="d")
plt.title(f"Confusion Matrix - {best_model_name} with Tuned Threshold")
plt.grid(False)
plt.show()


## 13. Predict Patient Condition and Seizure Probability

In [ ]:
# This helper function returns the final class label and seizure probability.
# The final class label is generated using the model's default prediction behavior.
def predict_patient_condition(model, patient_records):
    """
    Predict seizure condition and seizure probability for one or more patient records.

    Parameters
    ----------
    model : trained sklearn model or pipeline
        The trained classification model.
    patient_records : pandas DataFrame
        Patient records with the same feature columns used during training.

    Returns
    -------
    pandas DataFrame
        A table containing predicted class and seizure probability.
    """
    predicted_binary = model.predict(patient_records)
    seizure_probability = model.predict_proba(patient_records)[:, 1]
    predicted_label = [label_names[value] for value in predicted_binary]

    return pd.DataFrame({
        "predicted_label": predicted_label,
        "seizure_probability": seizure_probability
    })


# Demonstrate prediction on the first 10 test records.
sample_patients = X_test.iloc[:10].copy()
sample_actual = y_test.iloc[:10].map(label_names).reset_index(drop=True)

sample_predictions = predict_patient_condition(
    best_model,
    sample_patients
)

sample_predictions.insert(0, "actual_label", sample_actual)
display(sample_predictions)

## 14. Save the Final Model

In [ ]:
# Save the final model together with feature column order and label mapping.
# This makes future prediction safer because the feature order must remain the same.
model_artifact = {
    "model": best_model,
    "decision_method": "default_svm_predict",
    "feature_columns": feature_columns,
    "label_names": label_names,
    "best_model_name": best_model_name
}

output_model_path = Path("seizure_detection_model.joblib")
joblib.dump(model_artifact, output_model_path)

print(f"Model artifact saved to: {output_model_path.resolve()}")

## 15. Final Summary Tables

In [ ]:
# Final model comparison table.
display(metrics_df)

# Final selected model information.
final_summary = pd.DataFrame({
    "item": [
        "Selected model",
        "Target class",
    ],
    "value": [
        best_model_name,
        "Seizure = 1, Non-Seizure = 0",
    ]
})

final_summary
